# 📓 Week 22 — Continuous Learning

## Objective
After Pi deployment (Weeks 15–21), the detector generates logs
of predictions. This notebook:
1. Loads those logs (or simulates them)
2. Builds an incremental retraining dataset
3. Fine-tunes SecurityBERT on new data
4. Evaluates improvement

## Why continuous learning matters
```
Initial training   → Edge-IIoTset (2024 patterns)
Real deployment    → new attack variants emerge
Without retraining → model accuracy degrades
With this notebook → model stays up-to-date automatically
```

In [1]:
import warnings, json, time, math, random
from pathlib     import Path
from typing      import Dict, List, Optional, Tuple
from collections import defaultdict

import numpy  as np
import pandas as pd
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data     import Dataset, DataLoader, WeightedRandomSampler
from torch.optim          import AdamW
from transformers         import BertConfig, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics      import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot  as plt

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor':'#0f0f1a','axes.facecolor':'#16162a',
    'axes.edgecolor':'#444477','axes.labelcolor':'#ccccff',
    'xtick.color':'#aaaacc','ytick.color':'#aaaacc',
    'text.color':'#e0e0ff','grid.color':'#2a2a4a',
    'font.family':'DejaVu Sans','legend.facecolor':'#1a1a2e',
})
ACCENT='#7c6cfa'; GREEN='#4ecca3'; RED='#fc5c65'; YELLOW='#f7b731'
PALETTE=['#7c6cfa','#4ecca3','#f7b731','#fc5c65','#45aaf2',
         '#fd9644','#26de81','#a55eea','#2bcbba','#eb3b5a',
         '#20bf6b','#0fb9b1','#8854d0','#4b6584','#778ca3']

BASE_DIR    = Path('..')
CKPT_DIR    = BASE_DIR / 'checkpoints'
PROC_DIR    = BASE_DIR / 'data' / 'processed'
LOG_DIR     = BASE_DIR / 'logs'
OUT_DIR     = BASE_DIR / 'outputs' / 'figures'
REP_DIR     = BASE_DIR / 'outputs' / 'reports'

FINAL_CKPT  = CKPT_DIR / 'final_model.pt'         # NB08 base model
CL_CKPT     = CKPT_DIR / 'final_model_cl.pt'      # this notebook output
TOKEN_DATA  = PROC_DIR / 'tokenized_sequences.pt'

for d in [LOG_DIR, OUT_DIR, REP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Backdoor','DDoS_HTTP','DDoS_ICMP','DDoS_TCP','DDoS_UDP',
    'Fingerprinting','MITM','Normal','Password','Port_Scanning',
    'Ransomware','SQL_injection','Uploading','Vulnerability_scanner','XSS',
]
N_CLASSES  = 15
CLS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
torch.manual_seed(SEED); np.random.seed(SEED)

assert FINAL_CKPT.exists(), f'❌ {FINAL_CKPT} — run NB08 first'
assert TOKEN_DATA.exists(), f'❌ {TOKEN_DATA} — run NB04 first'
print('✅ Setup complete. Device:', DEVICE)

c:\Users\kaush\Desktop\LLM Threat Detection on IIOT\SecurityBERT CLAUDE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup complete. Device: cuda


## 📋 Step 2 — Simulate Deployment Logs

In real deployment (Week 21), the Pi writes logs like:
```json
{"timestamp":"2024-01-15T14:32:07","predicted":"DDoS_TCP",
 "confidence":0.94,"source_ip":"192.168.100.2",
 "action":"BLOCK_IP","verified":true,"true_label":"DDoS_TCP"}
```
We simulate 3 scenarios of model drift:
- **New attack variants** — DDoS packets with novel flag combinations
- **False positives** — Normal traffic misclassified as attacks
- **Low-confidence detections** — model uncertain on edge cases

In [2]:
def simulate_deployment_logs(
    n_total      : int = 2_000,
    drift_rate   : float = 0.15,
    rng_seed     : int = 99,
) -> List[Dict]:
    """
    Simulate deployment logs from Pi over 4 weeks of operation.

    Includes:
    - Normal correct detections (85%)
    - Low-confidence detections (8%)
    - False positives on normal traffic (4%)
    - Novel attack variants (3% — new patterns)
    """
    rng  = np.random.RandomState(rng_seed)
    logs = []
    t    = time.time() - 86400 * 28   # 4 weeks ago

    attack_weights = [0.05]*N_CLASSES
    attack_weights[CLASS_NAMES.index('DDoS_TCP')]  = 0.20
    attack_weights[CLASS_NAMES.index('DDoS_UDP')]  = 0.15
    attack_weights[CLASS_NAMES.index('Normal')]    = 0.25
    attack_weights[CLASS_NAMES.index('SQL_injection')] = 0.08
    total = sum(attack_weights)
    attack_weights = [w/total for w in attack_weights]

    for i in range(n_total):
        t      += rng.exponential(60)  # ~1 packet per minute
        true_cls= rng.choice(CLASS_NAMES, p=attack_weights)
        true_idx= CLS_TO_IDX[true_cls]

        # Simulate prediction with drift
        if rng.random() < drift_rate:
            # Drifted prediction — wrong class
            wrong_classes = [c for c in CLASS_NAMES if c != true_cls]
            pred_cls      = rng.choice(wrong_classes)
            conf          = float(rng.uniform(0.55, 0.75))
            verified      = True    # human verified the error
        else:
            pred_cls      = true_cls
            conf          = float(rng.uniform(0.72, 0.99))
            verified      = rng.random() < 0.30

        logs.append({
            'timestamp'    : time.strftime('%Y-%m-%dT%H:%M:%S',
                                           time.localtime(t)),
            'predicted'    : pred_cls,
            'true_label'   : true_cls if verified else None,
            'confidence'   : round(conf, 4),
            'source_ip'    : f'192.168.1.{rng.randint(10,250)}',
            'action'       : 'BLOCK_IP',
            'verified'     : verified,
            'sample_idx'   : i % 10000,   # index into tokenized_sequences
        })

    return logs


logs = simulate_deployment_logs(n_total=2_000, drift_rate=0.15)
print(f'✅ Simulated {len(logs):,} deployment logs.')

# Analyse logs
verified  = [l for l in logs if l['verified']]
errors    = [l for l in verified if l['predicted'] != l['true_label']]
error_rate= len(errors) / max(len(verified), 1)

print(f'   Verified logs  : {len(verified):,}')
print(f'   Errors found   : {len(errors):,}')
print(f'   Error rate     : {error_rate*100:.1f}%')
print(f'   Need retraining: {"✅ YES" if error_rate > 0.10 else "✅ NO"}')

# Save simulated logs
log_path = LOG_DIR / 'deployment_logs.json'
with open(log_path, 'w') as f:
    json.dump(logs, f, indent=2)
print(f'   Saved → {log_path}')

✅ Simulated 2,000 deployment logs.
   Verified logs  : 801
   Errors found   : 280
   Error rate     : 35.0%
   Need retraining: ✅ YES
   Saved → ..\logs\deployment_logs.json


## 🏗️ Step 3 — Build Retraining Dataset

In [3]:
# ── Load base tokenized data ───────────────────────────────────────────────────
data       = torch.load(TOKEN_DATA, weights_only=False)
input_ids  = data['input_ids'].long()
attn_masks = data['attention_mask'].long()
labels     = data['labels'].long()
label_map  = data['label_map']

# ── Build retraining dataset from verified logs ───────────────────────────────
# Strategy: use verified logs as high-priority samples
# + random sample from original data for class balance
RETRAIN_PER_CLASS    = 500    # from original data
VERIFIED_WEIGHT      = 3.0   # verified error samples get 3x weight

retrain_ids, retrain_masks, retrain_labels = [], [], []
sample_weights = []

print('🏗️  Building retraining dataset …')

# From original data — balanced
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_mask = (labels == cls_idx).nonzero(as_tuple=True)[0]
    n_take   = min(RETRAIN_PER_CLASS, len(cls_mask))
    sel      = cls_mask[torch.randperm(len(cls_mask), generator=
                        torch.Generator().manual_seed(SEED))[:n_take]]
    retrain_ids.append(input_ids[sel])
    retrain_masks.append(attn_masks[sel])
    retrain_labels.extend([cls_idx] * n_take)
    sample_weights.extend([1.0] * n_take)

# From verified error logs — higher weight
# Use sample_idx to pull exact tokenized sequences
error_logs = [l for l in logs if l['verified']
              and l['predicted'] != l['true_label']
              and l['true_label'] in CLS_TO_IDX]

for log_entry in error_logs[:500]:   # cap at 500
    idx     = log_entry['sample_idx'] % len(input_ids)
    true_cls= CLS_TO_IDX[log_entry['true_label']]
    retrain_ids.append(input_ids   [idx:idx+1])
    retrain_masks.append(attn_masks[idx:idx+1])
    retrain_labels.append(true_cls)
    sample_weights.append(VERIFIED_WEIGHT)

retrain_ids    = torch.cat(retrain_ids,   dim=0)
retrain_masks  = torch.cat(retrain_masks, dim=0)
retrain_labels = torch.tensor(retrain_labels, dtype=torch.long)
sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

print(f'✅ Retraining dataset built:')
print(f'   Total samples          : {len(retrain_labels):,}')
print(f'   From original data     : {RETRAIN_PER_CLASS * N_CLASSES:,}')
print(f'   From verified errors   : {len(error_logs[:500]):,}  (weight={VERIFIED_WEIGHT}×)')
print(f'   Seq len                : {retrain_ids.shape[1]}')

🏗️  Building retraining dataset …
✅ Retraining dataset built:
   Total samples          : 7,680
   From original data     : 7,500
   From verified errors   : 280  (weight=3.0×)
   Seq len                : 512


## 🔄 Step 4 — Incremental Fine-Tuning

In [4]:
# ── Model definition ──────────────────────────────────────────────────────────
class SecurityBERTWithSoftmax(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert       = BertModel(config, add_pooling_layer=True)
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.softmax    = nn.Softmax(dim=-1)

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.pooler_output)
        logits = self.classifier(pooled)
        return logits, self.softmax(logits)


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha.float())
        self.gamma = gamma
    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        loss = self.alpha[targets] * (1 - pt)**self.gamma * ce
        return loss.mean()


class CLDataset(torch.utils.data.Dataset):
    def __init__(self, ids, masks, labels):
        self.ids, self.masks, self.labels = ids, masks, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids':self.ids[i],
                'attention_mask':self.masks[i],
                'labels':self.labels[i]}


# ── Load base model ────────────────────────────────────────────────────────────
base_ckpt  = torch.load(FINAL_CKPT, map_location=DEVICE, weights_only=False)
config     = BertConfig(**base_ckpt['config'])
model      = SecurityBERTWithSoftmax(config).to(DEVICE)
model.load_state_dict(base_ckpt['model_state_dict'])

# ── Eval before fine-tuning ───────────────────────────────────────────────────
model.eval()
before_preds, before_true = [], []
val_ids   = input_ids [:2000].to(DEVICE)
val_masks = attn_masks[:2000].to(DEVICE)
val_labels= labels    [:2000]
with torch.no_grad():
    for i in range(0, 2000, 128):
        _, p = model(val_ids[i:i+128], val_masks[i:i+128])
        before_preds.extend(p.argmax(-1).cpu().numpy())
        before_true.extend(val_labels[i:i+128].numpy())
acc_before = accuracy_score(before_true, before_preds)
wf1_before = f1_score(before_true, before_preds, average='weighted',
                      zero_division=0)
print(f'📊 Before CL — Accuracy: {acc_before*100:.2f}%  WF1: {wf1_before:.4f}')

# ── Fine-tuning setup ──────────────────────────────────────────────────────────
CL_EPOCHS   = 2
CL_LR       = 5e-6       # very low LR — preserve original knowledge
CL_BATCH    = 64

# Compute class weights
cls_counts    = torch.bincount(retrain_labels, minlength=N_CLASSES).float()
cls_weights   = (len(retrain_labels) / (N_CLASSES * cls_counts.clamp(min=1)))

# Weighted sampler
sampler   = WeightedRandomSampler(
    sample_weights, len(retrain_labels), replacement=True
)
train_dataset = CLDataset(retrain_ids, retrain_masks, retrain_labels)
train_loader  = DataLoader(train_dataset, batch_size=CL_BATCH,
                           sampler=sampler, num_workers=0,
                           pin_memory=torch.cuda.is_available())

criterion = FocalLoss(cls_weights.to(DEVICE))
no_decay  = ['bias','LayerNorm.weight','LayerNorm.bias']
opt_params= [
    {'params': [p for n,p in model.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n,p in model.named_parameters()
                if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = AdamW(opt_params, lr=CL_LR, eps=1e-8)
total_steps = len(train_loader) * CL_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, int(total_steps*0.05), total_steps
)

# ── Fine-tuning loop ──────────────────────────────────────────────────────────
print(f'\n🔄 Incremental fine-tuning ({CL_EPOCHS} epochs, LR={CL_LR}) …')
cl_history = []

for epoch in range(1, CL_EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        ids  = batch['input_ids'].to(DEVICE)
        msk  = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        logits, _ = model(ids, msk)
        loss = criterion(logits, lbl)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        ep_loss += loss.item()
    avg_loss = ep_loss / len(train_loader)
    cl_history.append(avg_loss)
    print(f'   Epoch {epoch}/{CL_EPOCHS}  Loss: {avg_loss:.4f}')

# ── Eval after fine-tuning ────────────────────────────────────────────────────
model.eval()
after_preds = []
with torch.no_grad():
    for i in range(0, 2000, 128):
        _, p = model(val_ids[i:i+128], val_masks[i:i+128])
        after_preds.extend(p.argmax(-1).cpu().numpy())
acc_after = accuracy_score(before_true, after_preds)
wf1_after = f1_score(before_true, after_preds, average='weighted',
                     zero_division=0)

print(f'\n📊 Results:')
print(f'   Before CL: Accuracy={acc_before*100:.2f}%  WF1={wf1_before:.4f}')
print(f'   After  CL: Accuracy={acc_after*100:.2f}%   WF1={wf1_after:.4f}')
print(f'   Delta  : {(acc_after-acc_before)*100:+.2f}%  WF1 {wf1_after-wf1_before:+.4f}')

# Save updated model
torch.save({
    'model_state_dict': model.state_dict(),
    'config'          : config.to_dict(),
    'label_map'       : base_ckpt['label_map'],
    'cl_epochs'       : CL_EPOCHS,
    'cl_lr'           : CL_LR,
    'acc_before'      : float(acc_before),
    'acc_after'       : float(acc_after),
    'wf1_before'      : float(wf1_before),
    'wf1_after'       : float(wf1_after),
}, CL_CKPT)
print(f'\n💾 CL model saved → {CL_CKPT}')

📊 Before CL — Accuracy: 100.00%  WF1: 1.0000

🔄 Incremental fine-tuning (2 epochs, LR=5e-06) …
   Epoch 1/2  Loss: 0.6332
   Epoch 2/2  Loss: 0.5914

📊 Results:
   Before CL: Accuracy=100.00%  WF1=1.0000
   After  CL: Accuracy=99.60%   WF1=0.9980
   Delta  : -0.40%  WF1 -0.0020

💾 CL model saved → ..\checkpoints\final_model_cl.pt
